# Advanced Scikit-learn

In this notebook you learn to combine preprocessing and a model into one object, a pipeline, and to give different columns different preprocessing with a column transformer. A pipeline makes your code shorter and easier to reuse. It also makes cross-validation correct: when you cross-validate or tune a pipeline, every preprocessing step is fitted again in each fold, on the training part only. Without a pipeline it is easy to let the validation data influence the preprocessing, and the cross-validation score is then too optimistic. Section 2 shows how large this effect can be.

Outline

1. Pipelines
2. Pipelines and Cross-Validation
3. Column Transformers
4. Tuning a Pipeline
5. Exercise
6. Additional Concepts

**Working in Colab?** This notebook uses `optuna`, which Colab does not include. Run the cell below: it installs the package only when it detects Colab. Locally it does nothing, because the course environment already includes `optuna`.

In [1]:
# Installs the missing packages on Google Colab only; does nothing locally.
import sys

if "google.colab" in sys.modules:
    %pip install -q optuna

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 1. Pipelines

What is a `Pipeline` and why should we use one? Let's understand this by looking at an example. We will repeat the very first case from the [Data Preparation](../theme-2/01_data_prep.ipynb) notebook, where we scaled the data before training a support vector machine.

In [3]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler

In [4]:
X, y = fetch_california_housing(as_frame=True, return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

In [5]:
# how X_train looks before scaling
X_train.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
12069,4.2386,6.0,7.723077,1.169231,228.0,3.507692,33.83,-117.55
15925,4.3898,52.0,5.326622,1.100671,1485.0,3.322148,37.73,-122.44
11162,3.9333,26.0,4.668478,1.046196,1022.0,2.777174,33.83,-118.00
4904,1.4653,38.0,3.383495,1.009709,749.0,3.635922,34.01,-118.26
4683,3.1765,52.0,4.119792,1.043403,1135.0,1.970486,34.08,-118.36


In [6]:
# scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)

In [7]:
# how X_train looks after scaling
pd.DataFrame(X_train, columns=scaler.feature_names_in_)

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,0.190012,-1.795076,0.907714,0.149243,-1.047601,0.074082,-0.840062,1.003899
1,0.269311,1.855539,-0.042002,0.009186,0.052109,0.045276,0.985364,-1.434772
2,0.029895,-0.207852,-0.302824,-0.102099,-0.352955,-0.039332,-0.840062,0.779481
3,-1.264470,0.744482,-0.812064,-0.176636,-0.591794,0.093990,-0.755812,0.649818
4,-0.367016,1.855539,-0.520269,-0.107804,-0.254095,-0.164572,-0.723048,0.599947
...,...,...,...,...,...,...,...,...
16507,0.281216,-0.684019,0.224858,-0.103189,-0.624165,0.034252,1.238115,-0.846300
16508,-0.504949,-0.128491,-0.033022,-0.333310,-0.681031,0.010119,0.868349,-0.661778
16509,-0.355897,0.188954,-0.709927,-0.316025,-0.631164,-0.192383,0.447097,-1.165471
16510,0.920532,0.427037,-0.353503,-0.065518,-0.326709,-0.121509,-0.938355,0.814390


In [8]:
# train
model = SVR()
model.fit(X_train, y_train)

,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm.If none is given, 'rbf' will be used. If a callable is given it isused to precompute the kernel matrix.For an intuitive visualization of different kernel typessee :ref:`sphx_glr_auto_examples_svm_plot_svm_regression.py`",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.The penalty is a squared l2. For an intuitive visualization of theeffects of scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"epsilon epsilon: float, default=0.1Epsilon in the epsilon-SVR model. It specifies the epsilon-tubewithin which no penalty is associated in the training loss functionwith points predicted within a distance epsilon from the actualvalue. Must be non-negative.",0.1
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide <shrinking_svm>`.",True
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False
,"max_iter max_iter: int, default=-1Hard limit on iterations within solver, or -1 for no limit.",-1


In [9]:
# we cannot use X_test directly
# we must apply the same scaler that was fitted on X_train
X_test.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
14740,4.1518,22.0,5.663073,1.075472,1551.0,4.180593,32.58,-117.05
10101,5.7796,32.0,6.107226,0.927739,1296.0,3.020979,33.92,-117.97
20566,4.3487,29.0,5.930712,1.026217,1554.0,2.910112,38.65,-121.84
2670,2.4511,37.0,4.992958,1.316901,390.0,2.746479,33.20,-115.60
15709,5.0049,25.0,4.319261,1.039578,649.0,1.712401,37.79,-122.43


In [10]:
# if we skip scaling here
# the model will receive inputs in the wrong scale and perform poorly
y_pred = model.predict(np.array(X_test))
mae = mean_absolute_error(y_test, y_pred)
print(mae)

1.0047338322436465


In [11]:
# apply the scaler that was fitted on X_train
# you shouldn't fit a new scaler here, the scaling was already learned
X_test = scaler.transform(X_test)

In [12]:
# test
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print(mae)

0.38556904660770236


The issue here is the preprocessing and the model are separate pieces that you must manage yourself. Saving the model alone is not enough. When new data arrives, you must apply exactly the same fitted transformers, so you also need to save the scaler and any other preprocessing steps.

As your workflow grows, this becomes difficult to track, and you will eventually end up writing your own functions to bundle these steps together. There is also the problem that when everything is developed separately, each part is tuned in isolation while the rest is kept constant. In many cases, jointly optimising the entire chain leads to better results.

The most serious problem appears as soon as you cross-validate: each preprocessing step has to be fitted inside every fold, on the training part only. Section 2 shows what goes wrong otherwise.

Now let’s repeat the example using a pipeline.

In [13]:
# repeat
X, y = fetch_california_housing(as_frame=True, return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

In [14]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline(steps=[
                ('scaler', StandardScaler()),
                ('model', SVR())])

pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm.If none is given, 'rbf' will be used. If a callable is given it isused to precompute the kernel matrix.For an intuitive visualization of different kernel typessee :ref:`sphx_glr_auto_examples_svm_plot_svm_regression.py`",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0


Now these two steps are combined into a single pipeline. We fit the pipeline as one unit, so we no longer fit the scaler and the SVR separately. When making predictions, we also avoid scaling first and predicting second. We can pass the raw data directly into the pipeline and it will handle everything in the correct order. With only two steps the benefit may seem small, but as your preprocessing becomes more complex, the convenience of letting the pipeline manage the entire chain becomes obvious.

In [15]:
# We can use X_train without scaling
X_train.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
12069,4.2386,6.0,7.723077,1.169231,228.0,3.507692,33.83,-117.55
15925,4.3898,52.0,5.326622,1.100671,1485.0,3.322148,37.73,-122.44
11162,3.9333,26.0,4.668478,1.046196,1022.0,2.777174,33.83,-118.00
4904,1.4653,38.0,3.383495,1.009709,749.0,3.635922,34.01,-118.26
4683,3.1765,52.0,4.119792,1.043403,1135.0,1.970486,34.08,-118.36


In [16]:
# this runs all preprocessing steps and then fits the model in one call
pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](8,)","['MedInc','HouseAge','AveRooms',...,'AveOccup','Latitude','Longitude']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,8
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
Name,Type,Value


In [17]:
# we can now use X_test in its raw form
# the pipeline will handle the scaling internally instead of us doing it manually
X_test.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
14740,4.1518,22.0,5.663073,1.075472,1551.0,4.180593,32.58,-117.05
10101,5.7796,32.0,6.107226,0.927739,1296.0,3.020979,33.92,-117.97
20566,4.3487,29.0,5.930712,1.026217,1554.0,2.910112,38.65,-121.84
2670,2.4511,37.0,4.992958,1.316901,390.0,2.746479,33.20,-115.60
15709,5.0049,25.0,4.319261,1.039578,649.0,1.712401,37.79,-122.43


In [18]:
# the pipeline applies preprocessing and prediction in one step
y_pred = pipeline.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print(mae)

0.38556904660770236


Now that we have seen what a pipeline does and why it is useful, let’s look at its formal definition. The [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html) states:

>A sequence of data transformers with an optional final predictor. `Pipeline` allows you to sequentially apply a list of transformers to preprocess the data and, if desired, conclude the sequence with a final predictor for predictive modeling. Intermediate steps of the pipeline must be transformers, that is, they must implement `fit` and `transform` methods. The final estimator only needs to implement `fit`.

This means, a pipeline can be imagined as an ordered chain:

```
transformer1 → transformer2 → ... → transformerN → model (optional)
```

You can include as many transformers as you want, or even build a pipeline with only transformers and no model at the end.

The pipeline we built looks like this:

```
StandardScaler → SVR
```

which can be seen in the flowchart Jupyter Notebook creates.

<mark>YOUR TURN</mark>

Click on the flowchart and find the value of the hyperparameter `C` used in `SVR`.

`C` is the regularization strength in an SVM and it controls how closely the model tries to fit the training data. Regularization is a way to prevent overfitting by limiting how flexible the model can be. A large value of `C` allows the model to fit the training data very tightly, trying hard to avoid mistakes, while a small value of `C` encourages a smoother and simpler decision boundary by allowing more training errors.

In [19]:
# interact with the flowchart
pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](8,)","['MedInc','HouseAge','AveRooms',...,'AveOccup','Latitude','Longitude']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,8
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
Name,Type,Value


Let's get familiar with the pipeline instance.

In [20]:
# check the type of the pipeline object
type(pipeline)

sklearn.pipeline.Pipeline

In [21]:
# view all steps in the pipeline by their assigned names
pipeline.named_steps

{'scaler': StandardScaler(), 'model': SVR()}

In [22]:
# access a specific step in the pipeline by its name
pipeline['model']

,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm.If none is given, 'rbf' will be used. If a callable is given it isused to precompute the kernel matrix.For an intuitive visualization of different kernel typessee :ref:`sphx_glr_auto_examples_svm_plot_svm_regression.py`",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.The penalty is a squared l2. For an intuitive visualization of theeffects of scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"epsilon epsilon: float, default=0.1Epsilon in the epsilon-SVR model. It specifies the epsilon-tubewithin which no penalty is associated in the training loss functionwith points predicted within a distance epsilon from the actualvalue. Must be non-negative.",0.1
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide <shrinking_svm>`.",True
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False
,"max_iter max_iter: int, default=-1Hard limit on iterations within solver, or -1 for no limit.",-1


In [23]:
# number of input features the pipeline saw when it was fitted
pipeline.n_features_in_

8

In [24]:
# names of the input features the pipeline saw during fitting
pipeline.feature_names_in_

array(['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population',
       'AveOccup', 'Latitude', 'Longitude'], dtype=object)

In [25]:
# get all parameters of the pipeline
pipeline.get_params()

{'memory': None,
 'steps': [('scaler', StandardScaler()), ('model', SVR())],
 'transform_input': None,
 'verbose': False,
 'scaler': StandardScaler(),
 'model': SVR(),
 'scaler__copy': True,
 'scaler__with_mean': True,
 'scaler__with_std': True,
 'model__C': 1.0,
 'model__cache_size': 200,
 'model__coef0': 0.0,
 'model__degree': 3,
 'model__epsilon': 0.1,
 'model__gamma': 'scale',
 'model__kernel': 'rbf',
 'model__max_iter': -1,
 'model__shrinking': True,
 'model__tol': 0.001,
 'model__verbose': False}

Why do the parameter names look like `model__C` instead of just `C`?

Once we use a pipeline, we have several steps and each step has its own parameters. If we used just `C`, the pipeline would not know which step that parameter belongs to.

The documentation explains that the pipeline lets us set parameters for each step by using the step name, two underscores __, and then the parameter name.

>The purpose of the pipeline is to assemble several steps that can be cross-validated together while setting different parameters. For this, it enables setting parameters of the various steps using their names and the parameter name separated by a '__'.

Remember when we created the pipeline above, we used the step names `scaler` and `model`:

```python
pipeline = Pipeline(steps=[
                ('scaler', StandardScaler()),
                ('model', SVR())])
```

This means, if we want to set the `C` hyperparameter of `SVR`, we must refer to it as `model__C`. In other words, we always use `<step_name>__<parameter_name>` when working with pipeline parameters.

We can set parameters directly when building the pipeline by passing them into the steps, or we can update the parameters later using the pipeline’s parameter naming scheme.

In [26]:
# set C=10 directly when constructing the pipeline
pipeline = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('model', SVR(C=10))
])

<mark>YOUR TURN</mark>

Click on the flowchart and find the value of the hyperparameter `C` used in `SVR`. Verify that it is `10`.

In [27]:
pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.The penalty is a squared l2. For an intuitive visualization of theeffects of scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",10
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm.If none is given, 'rbf' will be used. If a callable is given it isused to precompute the kernel matrix.For an intuitive visualization of different kernel typessee :ref:`sphx_glr_auto_examples_svm_plot_svm_regression.py`",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'


or after construction by combining the step name and param name:

In [28]:
# or set C=10 after the pipeline is created
pipeline = Pipeline(steps=[
                ('scaler', StandardScaler()),
                ('model', SVR())])

pipeline = pipeline.set_params(model__C=10)

<mark>YOUR TURN</mark>

Click on the flowchart and find the value of the hyperparameter `C` used in `SVR`. Verify that it is `10`.

In [29]:
pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.The penalty is a squared l2. For an intuitive visualization of theeffects of scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",10
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm.If none is given, 'rbf' will be used. If a callable is given it isused to precompute the kernel matrix.For an intuitive visualization of different kernel typessee :ref:`sphx_glr_auto_examples_svm_plot_svm_regression.py`",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'


you can update the parameters of the pipeline during training by chaining

In [30]:
pipeline = Pipeline(steps=[
                ('scaler', StandardScaler()),
                ('model', SVR())])

# update and train
pipeline.set_params(model__C=10).fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](8,)","['MedInc','HouseAge','AveRooms',...,'AveOccup','Latitude','Longitude']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,8
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
Name,Type,Value


In [31]:
y_pred = pipeline.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print(mae)

0.3659403300940247


Note that the pipeline gives the same MAE as our manual approach with the same parameters. We are not changing how the model behaves. We are only using this approach to bundle several steps into one pipeline. (The last MAE is different only because we set `C=10`.)

<mark>YOUR TURN</mark>

Create a pipeline with [MinMaxScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html) and `SVR`. Use the default parameters while defining the pipeline. After creating the pipeline, update the following parameters:

- `MinMaxScaler`: set `feature_range` to `(-1, 1)`, the default is `(0, 1)`
- `SVR`: set `C` to `0.1`, the default is `1`.

Train and test this model on the housing dataset.

In [32]:
X, y = fetch_california_housing(as_frame=True, return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

In [33]:
# YOUR CODE HERE

You can save and load the pipelines using `joblib`, similar to saving and loading models.

So far so good, however notice that a transformer always applies its operation to the entire input. When our feature matrix contains different types of columns that need different preprocessing, this approach is not enough. To handle this properly, we will now introduce the `ColumnTransformer`.

## 2. Pipelines and Cross-Validation

In the [Linear Regression](../theme-3/02_linear_regression.ipynb) notebook you wrote the cross-validation loop yourself, because the scaler had to be fitted on the training part of each fold only. A pipeline does this for you. To see why it matters, we take an example where the effect is large.

The data below is pure noise: 100 samples with 10,000 random features, and random labels 0 and 1. No model can predict these labels better than chance, so an honest accuracy is about 0.5.

In [34]:
rng = np.random.default_rng(0) # fixed seed so that everyone gets the same data
X_noise = rng.normal(size=(100, 10000))
y_noise = rng.integers(0, 2, size=100)

With 10,000 features it is tempting to keep only the ones that are most related to the label, for example the 20 best according to `SelectKBest`, and then cross-validate a model on them:

In [35]:
from sklearn.feature_selection import SelectKBest
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# wrong: the feature selection sees all labels, also those of the validation folds
X_selected = SelectKBest(k=20).fit_transform(X_noise, y_noise)
scores = cross_val_score(LogisticRegression(), X_selected, y_noise, cv=5)
print(f"Accuracy: {scores.mean():.2f}")

Accuracy: 0.87


An accuracy of 0.87 on random labels. Among 10,000 random features, some agree with the labels by chance. `SelectKBest` found them using all 100 labels, those of the validation folds included, so every validation fold had already influenced which features the model gets. The cross-validation no longer measures how well the model predicts data it has not seen.

Now the same two steps in a pipeline:

In [36]:
pipeline = Pipeline(steps=[
    ('select', SelectKBest(k=20)),
    ('model', LogisticRegression())
])
scores = cross_val_score(pipeline, X_noise, y_noise, cv=5)
print(f"Accuracy: {scores.mean():.2f}")

Accuracy: 0.48


0.48, about the 0.5 that random labels allow. `cross_val_score` now fits the whole pipeline in each fold, on the training part only: first the feature selection, then the model. The validation fold plays no part in either.

The same holds for every step that learns something from the data: a scaler learns means and standard deviations, an imputer learns the values it fills in, an encoder learns the categories, a feature selection learns which features to keep. With a scaler the effect is usually small, with feature selection it can be as large as here. The rule is the same for all of them: every step that is fitted on data goes into the pipeline, and you cross-validate and tune the pipeline as a whole (section 4).

## 3. Column Transformers

Now let’s set up a problem with heterogeneous data, solve it once without pipelines, and then solve it again using pipelines with column transformers. To create this scenario, I will modify the housing dataset by turning the `HouseAge` column into a categorical feature `HouseAgeCat` and introducing some random missing values in the `MedInc` column. This will require us to use an ordinal encoder for `HouseAgeCat` and an imputer for `MedInc`.

In [37]:
X, y = fetch_california_housing(as_frame=True, return_X_y=True)

In [38]:
# turn HouseAge into a categorical feature named HouseAgeCat
# then drop the original numeric column
X['HouseAgeCat'] = pd.qcut(X['HouseAge'], q=3, labels=['new', 'mid', 'old'])
X = X.drop(columns=['HouseAge'])
X.head(n=10)

,MedInc,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,HouseAgeCat
0,8.3252,6.984127,1.023810,322.0,2.555556,37.88,-122.23,old
1,8.3014,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,new
2,7.2574,8.288136,1.073446,496.0,2.802260,37.85,-122.24,old
3,5.6431,5.817352,1.073059,558.0,2.547945,37.85,-122.25,old
4,3.8462,6.281853,1.081081,565.0,2.181467,37.85,-122.25,old
5,4.0368,4.761658,1.103627,413.0,2.139896,37.85,-122.25,old
6,3.6591,4.931907,0.951362,1094.0,2.128405,37.84,-122.25,old
7,3.1200,4.797527,1.061824,1157.0,1.788253,37.84,-122.25,old
8,2.0804,4.294118,1.117647,1206.0,2.026891,37.84,-122.26,old
9,3.6912,4.970588,0.990196,1551.0,2.172269,37.84,-122.25,old


In [39]:
# introduce missing values into the MedInc column (about 10 percent of the rows)
rng = np.random.default_rng(0) # fixed seed for reproducible results
mask = rng.random(len(X)) < 0.10
X.loc[mask, 'MedInc'] = np.nan
X.head(n=10)

,MedInc,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,HouseAgeCat
0,8.3252,6.984127,1.023810,322.0,2.555556,37.88,-122.23,old
1,8.3014,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,new
2,NaN,8.288136,1.073446,496.0,2.802260,37.85,-122.24,old
3,NaN,5.817352,1.073059,558.0,2.547945,37.85,-122.25,old
4,3.8462,6.281853,1.081081,565.0,2.181467,37.85,-122.25,old
5,4.0368,4.761658,1.103627,413.0,2.139896,37.85,-122.25,old
6,3.6591,4.931907,0.951362,1094.0,2.128405,37.84,-122.25,old
7,3.1200,4.797527,1.061824,1157.0,1.788253,37.84,-122.25,old
8,2.0804,4.294118,1.117647,1206.0,2.026891,37.84,-122.26,old
9,3.6912,4.970588,0.990196,1551.0,2.172269,37.84,-122.25,old


Now let us see how we might handle this without using pipelines. You have already seen encoding and imputation in the [Data Preparation](../theme-2/01_data_prep.ipynb) notebook.

In [40]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

In [41]:
# impute missing values in the MedInc column using the median
# fit the imputer on the training data and apply the same transformation to the test data
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy="median")
X_train[["MedInc"]] = imputer.fit_transform(X_train[["MedInc"]])
X_test[["MedInc"]] = imputer.transform(X_test[["MedInc"]])

In [42]:
# encode the categorical HouseAgeCat column using the defined order
# fit the encoder on the training data and apply the same transformation to the test data
from sklearn.preprocessing import OrdinalEncoder
encoder = OrdinalEncoder(categories=[['new', 'mid', 'old']], dtype='int')
X_train["HouseAgeCat"] = encoder.fit_transform(X_train[["HouseAgeCat"]])
X_test["HouseAgeCat"] = encoder.transform(X_test[["HouseAgeCat"]])

In [43]:
# MedInc now has no missing values
# HouseAgeCat is encoded as integers
X_train.head(n=10)

,MedInc,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,HouseAgeCat
12069,3.5464,7.723077,1.169231,228.0,3.507692,33.83,-117.55,0
15925,4.3898,5.326622,1.100671,1485.0,3.322148,37.73,-122.44,2
11162,3.5464,4.668478,1.046196,1022.0,2.777174,33.83,-118.00,1
4904,1.4653,3.383495,1.009709,749.0,3.635922,34.01,-118.26,2
4683,3.1765,4.119792,1.043403,1135.0,1.970486,34.08,-118.36,2
10796,9.2175,6.268734,0.927649,937.0,2.421189,33.62,-117.91,1
8839,3.5114,4.411348,1.078014,444.0,1.574468,34.08,-118.39,2
9033,8.7364,7.283113,1.212748,2750.0,2.276490,34.02,-118.66,1
12439,2.2141,4.375758,0.990909,2940.0,4.454545,33.67,-116.18,1
3094,1.8819,6.686493,1.464684,1802.0,2.232962,35.58,-118.45,0


Now let's train a `RandomForestRegressor`:

In [44]:
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(random_state=0)
model.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",0
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of

and test:

In [45]:
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print(mae)

0.3290115179748063


We obtained a reasonable MAE, so the approach works. Our next goal is to reproduce the same process using pipelines, which will require the `ColumnTransformer`.

From the [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html):

>Applies transformers to columns of an array or pandas DataFrame.
This estimator allows different columns or column subsets of the input to be transformed separately and the features generated by each transformer will be concatenated to form a single feature space. This is useful for heterogeneous or columnar data, to combine several feature extraction mechanisms or transformations into a single transformer.

This means, it creates a transformer that applies specific transformations to selected columns while passing the remaining columns through unchanged. This allows us to preprocess only the columns we choose without affecting the rest of the data.

In [46]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

In [47]:
# create a custom transformer that applies different preprocessing to different columns
# it takes X as input, transforms only the specified columns, and leaves the rest unchanged
# this can be plugged directly into a pipeline
from sklearn.compose import ColumnTransformer
ct = ColumnTransformer(
    transformers=[
        ("ordinal_enc", OrdinalEncoder(categories=[['new', 'mid', 'old']], dtype='int'), ["HouseAgeCat"]),
        ("imputer", SimpleImputer(), ["MedInc"])]
)

In [48]:
# now we can use it like any other transformer
# fit it on the training data and apply the transformations
ct.fit_transform(X_train)

array([[0.        , 3.87685885],
       [2.        , 4.3898    ],
       [1.        , 3.87685885],
       ...,
       [1.        , 3.1977    ],
       [1.        , 5.6315    ],
       [0.        , 1.3882    ]], shape=(16512, 2))

But there is a problem. We passed the entire `X_train` to the transformer, yet the output only contains the transformed columns. We want the full `X_train` back, with only the selected columns modified. To achieve this, we can use the `remainder` parameter, which tells the transformer what to do with the columns that are not explicitly listed.

In [49]:
ct = ColumnTransformer(
    transformers=[
        ("ordinal_enc", OrdinalEncoder(categories=[['new', 'mid', 'old']], dtype='int'), ["HouseAgeCat"]),
        ("imputer", SimpleImputer(), ["MedInc"])
    ],
    remainder='passthrough' # add this
)

In [50]:
ct.fit_transform(X_train)

array([[   0.        ,    3.87685885,    7.72307692, ...,    3.50769231,
          33.83      , -117.55      ],
       [   2.        ,    4.3898    ,    5.32662192, ...,    3.32214765,
          37.73      , -122.44      ],
       [   1.        ,    3.87685885,    4.66847826, ...,    2.77717391,
          33.83      , -118.        ],
       ...,
       [   1.        ,    3.1977    ,    3.64122137, ...,    1.7913486 ,
          36.58      , -121.9       ],
       [   1.        ,    5.6315    ,    4.54059829, ...,    2.24786325,
          33.62      , -117.93      ],
       [   0.        ,    1.3882    ,    3.9295302 , ...,    3.43624161,
          32.8       , -115.56      ]], shape=(16512, 8))

Now the full `X_train` is returned, with only the selected columns transformed and the rest left untouched. We can now place this column transformer inside a pipeline.

In [51]:
pipeline = Pipeline(steps=[
    ('preprocess', ct),
    ('model', RandomForestRegressor())
])

<mark>YOUR TURN</mark>

Inspect the pipeline flowchart, given below. Find the answers to the following, on the flowchart:

- What is the name of the column that goes into the imputer?
- What is the value of the `strategy` parameter in the imputer?

In [52]:
pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('ordinal_enc', ...), ('imputer', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, 

In [53]:
# you can access all the params
# including the ones inside the ct
pipeline.get_params()

{'memory': None,
 'steps': [('preprocess',
   ColumnTransformer(remainder='passthrough',
                     transformers=[('ordinal_enc',
                                    OrdinalEncoder(categories=[['new', 'mid',
                                                                'old']],
                                                   dtype='int'),
                                    ['HouseAgeCat']),
                                   ('imputer', SimpleImputer(), ['MedInc'])])),
  ('model', RandomForestRegressor())],
 'transform_input': None,
 'verbose': False,
 'preprocess': ColumnTransformer(remainder='passthrough',
                   transformers=[('ordinal_enc',
                                  OrdinalEncoder(categories=[['new', 'mid',
                                                              'old']],
                                                 dtype='int'),
                                  ['HouseAgeCat']),
                                 ('imputer', SimpleImpute

In our example without the pipelines, we used `random_state=0` and `strategy='median'`. We need to set these for our pipeline to follow the same steps. The MAE can still differ slightly (in the fourth decimal), because the `ColumnTransformer` puts the transformed columns first and a random forest with a fixed `random_state` depends on the column order.

<mark>YOUR TURN</mark>

Update the parameters of the pipeline, by setting `random_state=0` for the random forest and `strategy='median'` for the imputer. Inspect the flowchart to ensure the parameters were set correctly.

Solution will be given below.

In [54]:
# YOUR CODE HERE

In [55]:
# my solution
pipeline = pipeline.set_params(
    preprocess__imputer__strategy='median',
    model__random_state=0
)

pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('ordinal_enc', ...), ('imputer', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, 

In [56]:
pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](8,)","['MedInc','AveRooms','AveBedrms',...,'Latitude','Longitude','HouseAgeCat']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,8
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('ordinal_enc', ...), ('imputer', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset

In [57]:
y_pred = pipeline.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print(mae)

0.3287631211724807


In [58]:
pipeline.feature_names_in_

array(['MedInc', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup',
       'Latitude', 'Longitude', 'HouseAgeCat'], dtype=object)

## 4. Tuning a Pipeline

As a final step, let us see how a pipeline can be tuned. This is where _joint optimization_ becomes useful. For example, suppose we want to find out whether mean or median imputation works better. So far our workflow has been the following: run cross validation with mean imputation while keeping the model hyperparameters fixed, then run cross validation with median imputation while keeping the model hyperparameters fixed, and pick the better one. After that, we would tune the model hyperparameters while keeping the chosen strategy fixed.

With a pipeline, we no longer need to separate these steps. We can tune the entire pipeline at once and let cross validation jointly select both the imputation strategy and the model hyperparameters.

In [59]:
import optuna
from sklearn.base import clone
from sklearn.model_selection import cross_validate

In [60]:
# recreate the pipeline as before
# this time we set random_state=0 directly when defining the model
pipeline = Pipeline(steps=[
    ('preprocess', ct),
    ('model', RandomForestRegressor(random_state=0))
])

In [61]:
# now we tune the entire pipeline
# the process is the same as tuning a model, but parameter names include the step name
# for example: model__n_estimators or preprocess__imputer__strategy
def objective(trial):
    params = {
        "model__n_estimators": trial.suggest_int("model__n_estimators", 50, 200),
        "model__min_samples_split": trial.suggest_int("model__min_samples_split", 2, 10),
        "preprocess__imputer__strategy": trial.suggest_categorical("preprocess__imputer__strategy", ["mean", "median"])
    }

    # a fresh copy of the pipeline with these params (the original pipeline stays unchanged)
    p = clone(pipeline).set_params(**params)

    # cross validate
    result = cross_validate(p, X_train, y_train, cv=3, scoring='neg_mean_absolute_error', n_jobs=-1)
    mae = np.mean(result['test_score'])*-1

    return mae

In [62]:
# a seeded sampler makes the search reproducible, like random_state for a model
# this takes a minute or two
study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=0))
study.optimize(objective, n_trials=10)

[I 2026-11-14 10:43:26,635] A new study created in memory with name: no-name-9d242c71-42bb-4bba-9ec6-71b1dba48c61


[I 2026-11-14 10:43:31,869] Trial 0 finished with value: 0.3431451847601348 and parameters: {'model__n_estimators': 132, 'model__min_samples_split': 8, 'preprocess__imputer__strategy': 'mean'}. Best is trial 0 with value: 0.3431451847601348.


[I 2026-11-14 10:43:36,062] Trial 1 finished with value: 0.3424707161702663 and parameters: {'model__n_estimators': 113, 'model__min_samples_split': 7, 'preprocess__imputer__strategy': 'median'}. Best is trial 1 with value: 0.3424707161702663.


[I 2026-11-14 10:43:42,258] Trial 2 finished with value: 0.34141231970417363 and parameters: {'model__n_estimators': 195, 'model__min_samples_split': 5, 'preprocess__imputer__strategy': 'mean'}. Best is trial 2 with value: 0.34141231970417363.


[I 2026-11-14 10:43:46,325] Trial 3 finished with value: 0.3432976247100769 and parameters: {'model__n_estimators': 135, 'model__min_samples_split': 10, 'preprocess__imputer__strategy': 'median'}. Best is trial 2 with value: 0.34141231970417363.


[I 2026-11-14 10:43:48,838] Trial 4 finished with value: 0.3447547128265655 and parameters: {'model__n_estimators': 53, 'model__min_samples_split': 9, 'preprocess__imputer__strategy': 'median'}. Best is trial 2 with value: 0.34141231970417363.


[I 2026-11-14 10:43:55,473] Trial 5 finished with value: 0.3422890571222026 and parameters: {'model__n_estimators': 197, 'model__min_samples_split': 9, 'preprocess__imputer__strategy': 'median'}. Best is trial 2 with value: 0.34141231970417363.


[I 2026-11-14 10:43:58,259] Trial 6 finished with value: 0.3435387713016009 and parameters: {'model__n_estimators': 67, 'model__min_samples_split': 7, 'preprocess__imputer__strategy': 'median'}. Best is trial 2 with value: 0.34141231970417363.


[I 2026-11-14 10:44:02,366] Trial 7 finished with value: 0.341076736482991 and parameters: {'model__n_estimators': 128, 'model__min_samples_split': 5, 'preprocess__imputer__strategy': 'median'}. Best is trial 7 with value: 0.341076736482991.


[I 2026-11-14 10:44:06,009] Trial 8 finished with value: 0.3422509933284736 and parameters: {'model__n_estimators': 118, 'model__min_samples_split': 7, 'preprocess__imputer__strategy': 'median'}. Best is trial 7 with value: 0.341076736482991.


[I 2026-11-14 10:44:10,393] Trial 9 finished with value: 0.3426384372650089 and parameters: {'model__n_estimators': 142, 'model__min_samples_split': 7, 'preprocess__imputer__strategy': 'mean'}. Best is trial 7 with value: 0.341076736482991.


In [63]:
print("Best params:", study.best_params)
print("Best mae:", study.best_value)

Best params: {'model__n_estimators': 128, 'model__min_samples_split': 5, 'preprocess__imputer__strategy': 'median'}
Best mae: 0.341076736482991


In [64]:
# apply the best parameters found by Optuna
# you can verify in the flowchart that n_estimators matches the selected value
pipeline.set_params(**study.best_params)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('ordinal_enc', ...), ('imputer', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, 

In [65]:
pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](8,)","['MedInc','AveRooms','AveBedrms',...,'Latitude','Longitude','HouseAgeCat']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,8
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('ordinal_enc', ...), ('imputer', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset

In [66]:
y_pred = pipeline.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print(mae)

0.3294030049697732


## 5. Exercise

<mark>YOUR TURN</mark>

Create a pipeline with [MinMaxScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html) and `SVR`. Tune the pipeline on the housing dataset.


In [67]:
X, y = fetch_california_housing(as_frame=True, return_X_y=True)

In [68]:
# YOUR CODE HERE

## 6. Additional Concepts

Scikit-learn is a large library and there is not enough time to cover every idea in detail. As a data scientist, it is important to become comfortable learning new concepts when they are needed, because it is impossible to know everything in advance. Below is a short list of topics worth being aware of. You do not need to master them now, but it helps to know they exist so that you can return to them later when they become relevant.

- [Pipelines and composite estimators](https://scikit-learn.org/stable/modules/compose.html#combining-estimators)
- Combining predictors using [stacking](https://scikit-learn.org/stable/auto_examples/ensemble/plot_stack_predictors.html)
- [Semi-supervised learning](https://scikit-learn.org/stable/modules/semi_supervised.html)
- [Anomaly detection](https://scikit-learn.org/stable/modules/outlier_detection.html)
- Data generation ([make_classification](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_classification.html), [make_regression](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_regression.html) etc.)
- [Plotting calibration curves](https://scikit-learn.org/stable/modules/generated/sklearn.calibration.CalibrationDisplay.html)
- [Nested cross-validation](https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html)
- and take a look at the [User Guide](https://scikit-learn.org/stable/user_guide.html) for more.